# 🎓 MPCIM Thesis - Complete Analysis

**Dual-Dimensional Predictive Analytics untuk Career Progression**

Author: Deni Sulaeman | November 2025

## 1. Setup

In [ ]:
# Install packages (uncomment if needed)
# !pip install pandas numpy matplotlib seaborn plotly scikit-learn xgboost shap imbalanced-learn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")
print("✅ Libraries imported!")

## 2. Load Data

Upload your CSV file or specify the path to your data file.

In [ ]:
import os
from pathlib import Path

# Get the notebook directory and project root
notebook_dir = Path(os.getcwd())
if 'notebooks' in str(notebook_dir):
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

# Construct absolute path to data
data_path = project_root / "data" / "processed" / "full_dataset_processed.csv"

# Check if file exists
if data_path.exists():
    data_file = str(data_path)
    print(f"✅ Data file found: {data_file}")
else:
    # Fallback paths
    for path in [Path("../data/processed/full_dataset_processed.csv"), 
                 Path("data/processed/full_dataset_processed.csv")]:
        if path.exists():
            data_file = str(path.resolve())
            print(f"✅ Data file found: {data_file}")
            break
    else:
        print("❌ Data file not found!")
        # For Google Colab
        # from google.colab import files
        # uploaded = files.upload()
        # data_file = list(uploaded.keys())[0]
        raise FileNotFoundError("Data file not found")

## 3. Load & Explore

In [ ]:
df = pd.read_csv(data_file)
print(f"Shape: {df.shape}")
display(df.head())
display(df.describe())

## 4. Data Preparation

In [ ]:
y = df["has_promotion"]
X = df.drop(columns=["has_promotion", "employee_id_hash"], errors="ignore")
X = X.select_dtypes(include=[np.number])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)
print(f"✅ Training: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

## 5. Model Training

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train_bal)
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

## 6. Feature Importance

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1][:10]
plt.figure(figsize=(10, 6))
plt.barh(range(10), importances[indices])
plt.yticks(range(10), [X.columns[i] for i in indices])
plt.xlabel("Importance")
plt.title("Top 10 Features")
plt.gca().invert_yaxis()
plt.show()

## 7. Done!

✅ Analysis complete. Review results above.